In [62]:
import torch
import numpy as np
import pandas as pd
import einops as eops
import torch.nn as nn

In [27]:
import kagglehub
import pathlib

In [24]:
path = kagglehub.dataset_download("emirhanakku/synthetic-medical-triage-priority-dataset")


100%|███████████████████████████████████████████████████████████████████████████████| 228k/228k [00:00<00:00, 259kB/s]

Extracting files...


In [25]:
path = "\\".join(path.split('\\')[-2:])

In [35]:
files = pathlib.Path(path).glob("*.*")

In [36]:
dataset = list(files)[0]

In [55]:
data = pd.read_csv(dataset)

In [59]:
data


,age,heart_rate,systolic_blood_pressure,oxygen_saturation,body_temperature,pain_level_1,pain_level_2,pain_level_3,pain_level_4,pain_level_5,...,previous_er_visits_6,previous_er_visits_7,previous_er_visits_8,previous_er_visits_9,previous_er_visits_10,previous_er_visits_11,triage_level_0,triage_level_1,triage_level_2,triage_level_3
0,-1.403914,0.713711,1.011374,0.394741,-0.813392,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,1.805332,3.808426,1.622645,-0.025574,2.323294,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,0.334210,0.224451,0.006764,0.724990,0.563690,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
3,0.632622,0.082978,1.016690,-1.076363,0.356034,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
4,-0.288793,-1.490905,-1.077576,0.875102,-1.053835,0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17995,0.423210,0.990761,0.899751,-0.205709,0.651123,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
17996,-0.398734,0.088873,-0.003866,0.574877,-0.201356,0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,0
17997,0.758270,-1.084171,-1.327400,-0.656048,-0.736887,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
17998,1.344625,0.430765,0.851912,0.004448,0.935283,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,0


In [57]:
OnehotCol = ['pain_level', 'arrival_mode', 'chronic_disease_count', 'previous_er_visits', 'triage_level']
data = pd.get_dummies(data, columns=OnehotCol, dtype=int)

normCol = ['age', 'heart_rate', 'systolic_blood_pressure', 'oxygen_saturation', 'body_temperature']
for col in normCol:
    data[col] = (data[col] - data[col].mean())/ data[col].std()

In [63]:
class AmbSelModel(nn.Module):
    def __init__(self, _nf :int):
        super(AmbSelModel, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(_nf, 64),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(p=0.25),
            nn.Linear(128, 128),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=0.25),
            nn.Softmax(3)
        )

    def forward(self, inp):
        return self.model(inp)

In [65]:
class Train():
    def __init__(self, _nFeature :int):
        self.model = AmbSelModel(_nFeature)
        self.loss = nn.CrossEntropyLoss()
        self.optim = torch.optim.Adam(self.model.parameters(), lr=0.01)

    def train(self, X :np.ndarray, Y :np.ndarray, epochs:int=100, _nbatch:int=32):
        print("Training starts")
        model = self.model
        optim = self.optim
        loss_fn = self.loss

        Y = np.expand_dims(Y, -1)
        X = torch.from_numpy(X).float()
        Y = torch.from_numpy(Y).float()

        inp = TensorDataset(X, Y)
        print(type(inp), inp)
        data_loader = DataLoader(inp, batch_size=_nbatch, shuffle=True)
            
        for epoch in range(epochs):
            model.train()

            for i, (train_X, train_Y) in enumerate(data_loader):
                optim.zero_grad()
                logits = model(train_X)
                logit_loss = loss_fn(logits, train_Y)
                logit_loss.backward()
                optim.step()

            with torch.no_grad():
                idx = epoch%X.shape[0]
                pred = model(X[idx:idx+1])
                loss = loss_fn(pred, Y[idx:idx+1])
                print(f"Finishing epoch: {epoch} with {loss.item()*100}% logit loss")

#simple softmax between triage-level
    def pred(self, inp :np.ndarray, k :int=3):
        with torch.no_grad():
            inp = torch.Tensor(inp)
            pred_score = self.model(inp)
            idx = []
            print(pred_score)
            for i in range(len(pred_score)):
                arg = torch.argmax(pred_score[i])
                pred_score[i] = arg+1

            return pred_score

    def save(self, name :str):
        self.model.save(name)